# Lecture 1: Entropy, information, and reversible dynamics

The lecture introduces probability, reversible state-update rules, phase space, Liouville’s theorem, and entropy.

## Exercise: Reversible versus irreversible maps

Represent a system by one of N=32 discrete states.

Implement three update rules:

A permutation of the states

A many-to-one rule such as

$x_{n+1} = \lfloor {x_n / 2} \rfloor$

A reversible modular rule

$x_{n+1} = (5x_n + 1) % 32$

Start with a probability distribution p_i, evolve it, and calculate

$S = − \sum_i p_i ln(p_i)$

## Questions:

Which maps preserve distinguishability of initial states?
Which map destroys information?
Does fine-grained entropy change under a permutation?
What happens if you group the 32 states into four coarse macrostates?

### The important discovery should be:

Reversible microscopic evolution preserves fine-grained information, while coarse-graining can produce increasing entropy.

In [ ]:
import math
import numpy as np
from scipy.stats import entropy

# This will be available to all random functions below
rng = np.random.default_rng()

def is_valid_distribution(distn, atol=1e-8):
    return np.all(distn >= 0) and np.isclose(np.sum(distn), 1.0, atol=atol)

def generate_uniform_distribution(length):
    v = np.full(length, 1.0 / length)
    return v

def generate_random_distribution(length):
    v = rng.uniform(0.0, 1.0, size=length)
    return v / np.sum(v)

def generate_gaussian_distribution(length, mu=0.0, sigma=1.0):
    # Choosing -3 to +3 standard deviations captures ~99.7% of the curve
    x = np.linspace(mu - 3*sigma, mu + 3*sigma, num=length)
    # Normalize
    raw_weights = np.exp(-0.5 * ((x - mu) / sigma) ** 2)
    return raw_weights / np.sum(raw_weights)



# In this update, state x_(n+1) = floor(x_n / 2) - 
# in other words, the microstates all accumulate at the low indices
def update_distn_with_floor(distn):
    new_distn = np.full(distn.size, 0.0)
    for index,val in enumerate(distn):
        new_index = math.floor(index / 2)
        new_distn[new_index] += val
    return new_distn

# In this update, state x_(n+1) = (5 * x_n + 1) % length - 
# it is a fixed reversible permutation at least for length 32, or anything where 5 is a relative prime
def update_distn_with_fixed_permutation(distn):
    new_distn = np.full(distn.size, 0.0)
    for index,val in enumerate(distn):
        new_index = (5 * index + 1) % distn.size
        new_distn[new_index] += val
    return new_distn

# This is also a permutation, but not fixed in this case (still reversible)
def update_distn_random_permutation(distn):
    return rng.permutation(distn)


# Just use mod to divide the fine-grained distribution into coarse states
def compute_coarse_grained_entropy(distn, num_coarse_states):
    coarse_distn = np.full(num_coarse_states, 0.0)
    for index,val in enumerate(distn):
        coarse_distn[index % num_coarse_states] += val
    return entropy(coarse_distn)



In [ ]:
def print_if_valid(distn):
    if is_valid_distribution(distn):
        print('Entropy is: ', entropy(distn))
    else:
        print('Invalid distribution')

#distn = generate_uniform_distribution(32)
distn = generate_gaussian_distribution(32)
#distn = generate_random_distribution(32)

print(distn)
print_if_valid(distn)
for i in range(1, 10):
    #distn = update_distn_with_floor(distn)
    distn = update_distn_random_permutation(distn)
    #distn = update_distn_with_fixed_permutation(distn)

    print_if_valid(distn)

    # This is optional for exploring the effect of coarse-graining
    c_entropy = compute_coarse_grained_entropy(distn, 4)
    print('Coarse-grained entropy is: ', c_entropy)
